## This is the code to train the model and acquire influence for Number of Samples Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [1]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [3]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [4]:
import random
from keras.optimizers import SGD

In [5]:
from sklearn.datasets import make_classification

In [6]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to test on different number of samples.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [7]:
train_pool = 789
test_size = 100
train_sizes=[400,600]
n_features=8
seed=42
# sep = 5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [8]:
df = pd.read_csv('train.csv')

In [9]:
df = df.copy()
df.rename(columns={"PassengerId": "ID"}, inplace=True)
df = df.drop(columns=["Name", "Ticket", "Cabin", "ID"])


In [10]:
df = df.dropna(subset=["Embarked"])

In [11]:
df['MissAge'] = df['Age'].isna().astype(int)

num_missing = df['MissAge'].sum()
percent_missing = df['MissAge'].mean() * 100

print(f"Missing Age: {num_missing} samples ({percent_missing:.2f}%)")

Missing Age: 177 samples (19.91%)


In [12]:
# df = df.copy()
# df['Age'] = df['Age'].fillna(
#     df.groupby(['Pclass', 'Sex'])['Age'].transform('median')
# )

In [14]:
df = df.copy()
df['Age'] = df['Age'].fillna(
    df.groupby(['Pclass', 'Sex'])['Age'].transform('mean')
)
round_imputed_age = True

if round_imputed_age:
    df["Age"] = np.where(
        df["MissAge"] == 1,      # imputed indicator
        np.round(df["Age"]),
        df["Age"]
    )


In [15]:
sex_trans = LabelEncoder()
df["Sex"] = sex_trans.fit_transform(df["Sex"])

Emb_trans = LabelEncoder()
df["Embarked"] = sex_trans.fit_transform(df["Embarked"])

In [16]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [17]:
print(X)

     Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  MissAge
0         3    1  22.0      1      0   7.2500         2        0
1         1    0  38.0      1      0  71.2833         0        0
2         3    0  26.0      0      0   7.9250         2        0
3         1    0  35.0      1      0  53.1000         2        0
4         3    1  35.0      0      0   8.0500         2        0
..      ...  ...   ...    ...    ...      ...       ...      ...
886       2    1  27.0      0      0  13.0000         2        0
887       1    0  19.0      0      0  30.0000         2        0
888       3    0  22.0      1      2  23.4500         2        1
889       1    1  26.0      0      0  30.0000         0        0
890       3    1  32.0      0      0   7.7500         1        0

[889 rows x 8 columns]


3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [18]:
df = pd.DataFrame(X.to_numpy(), columns=[f'feature_{i+1}' for i in range(n_features)])
df['label'] = y.to_numpy()
df['id'] = np.arange(1, len(df) + 1)
print(df)

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0          3.0        1.0       22.0        1.0        0.0     7.2500   
1          1.0        0.0       38.0        1.0        0.0    71.2833   
2          3.0        0.0       26.0        0.0        0.0     7.9250   
3          1.0        0.0       35.0        1.0        0.0    53.1000   
4          3.0        1.0       35.0        0.0        0.0     8.0500   
..         ...        ...        ...        ...        ...        ...   
884        2.0        1.0       27.0        0.0        0.0    13.0000   
885        1.0        0.0       19.0        0.0        0.0    30.0000   
886        3.0        0.0       22.0        1.0        2.0    23.4500   
887        1.0        1.0       26.0        0.0        0.0    30.0000   
888        3.0        1.0       32.0        0.0        0.0     7.7500   

     feature_7  feature_8  label   id  
0          2.0        0.0      0    1  
1          0.0        0.0      1    2  
2  

In [19]:
orgin_df = df.copy()

In [20]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [21]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0        3.0        1.0       22.0        1.0        0.0     7.2500   
1        1.0        0.0       38.0        1.0        0.0    71.2833   
2        3.0        0.0       26.0        0.0        0.0     7.9250   
3        1.0        0.0       35.0        1.0        0.0    53.1000   
4        3.0        1.0       35.0        0.0        0.0     8.0500   

   feature_7  feature_8  label  id  
0        2.0        0.0      0   1  
1        0.0        0.0      1   2  
2        2.0        0.0      1   3  
3        2.0        0.0      1   4  
4        2.0        0.0      0   5  
   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0        3.0        1.0       27.0        0.0        0.0     7.7500   
1        2.0        1.0       16.0        0.0        0.0    26.0000   
2        3.0        0.0       22.0        8.0        2.0    69.5500   
3        1.0        1.0       41.0        0.0        0.0    30.6958

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [22]:
features_to_test = n_features
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

**The most important thing in this experiment is the following code**:  
Based on the train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000] defined above, we could map the number of training samples with the following code. By choosing the number in [], we could modify the train set size. Therefore, only changing the following code block is enough to produce the experiment result successfully.

In [23]:
train_df = nested_train_dfs[1]

In [24]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train.shape)

(600, 9)


In [25]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)

(100, 9)


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [26]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [27]:
# D = pairwise_distances(X_all) 

In [28]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [29]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [30]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [31]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

3/3 - 1s - loss: 0.7011 - accuracy: 0.5467 - val_loss: 0.8569 - val_accuracy: 0.4400 - 639ms/epoch - 213ms/step
3/3 - 0s - loss: 0.6980 - accuracy: 0.5517 - val_loss: 0.7562 - val_accuracy: 0.5000 - 34ms/epoch - 11ms/step
3/3 - 0s - loss: 0.6922 - accuracy: 0.5517 - val_loss: 0.7026 - val_accuracy: 0.5400 - 26ms/epoch - 9ms/step
3/3 - 0s - loss: 0.6800 - accuracy: 0.5767 - val_loss: 0.6669 - val_accuracy: 0.5600 - 29ms/epoch - 10ms/step
3/3 - 0s - loss: 0.6728 - accuracy: 0.5783 - val_loss: 0.6399 - val_accuracy: 0.6100 - 26ms/epoch - 9ms/step
3/3 - 0s - loss: 0.6668 - accuracy: 0.5817 - val_loss: 0.6169 - val_accuracy: 0.6500 - 28ms/epoch - 9ms/step
3/3 - 0s - loss: 0.6627 - accuracy: 0.5917 - val_loss: 0.6043 - val_accuracy: 0.6700 - 26ms/epoch - 9ms/step
3/3 - 0s - loss: 0.6592 - accuracy: 0.6000 - val_loss: 0.5962 - val_accuracy: 0.6700 - 28ms/epoch - 9ms/step
3/3 - 0s - loss: 0.6561 - accuracy: 0.6033 - val_loss: 0.5899 - val_accuracy: 0.6700 - 31ms/epoch - 10ms/step
3/3 - 0s - lo

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [32]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [33]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

     Train_ID     Score
0           1  0.496887
1           2  0.153170
2           3 -0.190163
3           4  0.105631
4           5  0.171192
..        ...       ...
595       596  0.105631
596       597  0.084106
597       598  0.244095
598       599  0.102802
599       600  0.026618

[600 rows x 2 columns]


In [34]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

     Train_ID     Score
0           1  0.063343
1           2 -0.060326
2           3 -0.205470
3           4 -0.056439
4           5  0.096198
..        ...       ...
595       596 -0.056439
596       597  0.018357
597       598  0.070577
598       599 -0.060226
599       600 -0.085322

[600 rows x 2 columns]


Influence Function Plot

In [35]:
# N = 150
# plt.figure(figsize=(10, 5))
# plt.plot(df["Train_ID"][:N],df["Score"][:N],label="Influence function Data Influence")
# plt.plot(TracIn_df["Train_ID"][:N],TracIn_df["Score"][:N],label="TracIn Data Influence", color="tab:orange")
# plt.xlabel("Train ID",fontsize=16)
# plt.ylabel("Influence Score",fontsize=16)
# plt.title("Influence Function vs TracIn Influence Score",fontsize=20)

# plt.xticks(fontsize=14)
# plt.yticks(fontsize=14)

# plt.legend()
# plt.grid()
# plt.savefig("if_tc_first_150_600_new_MissAgeImput.png", dpi=300, bbox_inches="tight")
# plt.show()

In [36]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)
TracIn_sorted.to_csv("TC_600_new_MeanAgeImput_Rounded_pclasswithsex_new.csv",index = False)
df_sorted.to_csv("IF_600_new_MeanAgeImput_Rounded_pclasswithsex_new.csv",index = False)

2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [37]:
k =600

# Top-k IF (absolute)
topk_if = df_sorted.iloc[
    df_sorted['Score'].sort_values(ascending=False).index[:k]
]

# Map back to original dataframe
topk_if_rows = (
    orgin_df.set_index('id')
      .loc[topk_if['Train_ID']]
      .reset_index()
)
print(topk_if)
print(topk_if_rows)
# Count missing
missing_if = topk_if_rows['feature_8'].sum()
percent_if = (missing_if / k) * 100

print(f"\n[IF] Top-{k} missing Age: {missing_if}/{k} ({percent_if:.2f}%)")

     Train_ID     Score
0         483  1.642138
1         258  1.209735
2          16  0.846183
3         477  0.563332
4         371  0.538838
..        ...       ...
595         9 -1.769925
596       267 -1.788267
597       271 -2.003197
598        68 -2.540614
599        85 -2.824638

[600 rows x 2 columns]
      id  feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0    483        3.0        0.0       63.0        0.0        0.0     9.5875   
1    258        1.0        0.0       35.0        0.0        0.0   512.3292   
2     16        2.0        0.0       55.0        0.0        0.0    16.0000   
3    477        3.0        1.0       29.0        1.0        0.0     7.0458   
4    371        3.0        1.0       18.0        1.0        0.0     6.4958   
..   ...        ...        ...        ...        ...        ...        ...   
595    9        3.0        0.0       27.0        0.0        2.0    11.1333   
596  267        3.0        1.0       25.0        1.0        0.0 

In [38]:
topk_if_rows.to_csv("IF_600_new_MeanAgeImput_Rounded_pclasswithsex_OriginalData_new.csv",index = False)